In [0]:
%sql

select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details limit 100;

select count(distinct Document_Id) from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details
where cluster is not null
and upper(trim(Document_Id)) RLIKE '^[0-9]+$' 
-- order by cluster,

In [0]:
%sql
    
----------- Creating review for user access, using Oct2025 Audit data------------
CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis as
select cast(CMS.Document_Id as string) as Doc_ID, (case when S1.document_id is not null then true else false end) as scheduled, CL.cluster, upper(trim(UA.UserName)) as UserName, UA.Access_TS, upper(trim(UP.DisplayName)) as DisplayName, upper(trim(UP.Title)) as Title, upper(trim(UP.BU)) as BU, upper(trim(UP.BU_Old)) as BU_Old, upper(trim(UP.Country))as Country, UA.event_type, upper(trim(CMS.updated)) as last_update_by, upper(trim(CMS.updated)) as Owner
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as CMS
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details as CL
on upper(trim(CL.Document_ID))=upper(trim(cast(CMS.Document_Id as string)))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA
on upper(trim(CMS.Document_ID)) = upper(trim(CAST(UA.WEBI_Doc_ID AS string)))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted as S1
on upper(trim(CMS.Document_Id))=upper(trim(S1.document_id))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP 
on upper(trim(UP.UserName)) = upper(trim(UA.UserName))
where upper(trim(cms.Document_name)) not in ('DOCUMENT NOT FOUND ON SERVER');

----------- Creating review for report ownership------------
CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis as
select distinct 
  cast(CMS.Document_Id as string) as Doc_ID, 
  (case when S1.document_id is not null then true else false end) as scheduled, 
  CL.cluster, 
  upper(trim(
    case 
      when upper(trim(CMS.created)) = 'ADMINISTRATOR' then CMS.lastAuthor 
      else CMS.created 
    end
  )) as Owner, 
  upper(trim(
    case 
      when UP.DisplayName is null then 'UNIDENTIFIED' 
      else UP.DisplayName 
    end
  )) as OwnerName, 
  upper(trim(
    case 
      when UP.Title is null then 'UNIDENTIFIED' 
      else UP.Title 
    end
  )) as OwnerTitle, 
  upper(trim(
    case 
      when UP.BU is null then 'UNIDENTIFIED' 
      else UP.BU 
    end
  )) as OwnerBU, 
  upper(trim(
    case 
      when UP.BU_Old is null then 'UNIDENTIFIED' 
      else UP.BU_Old 
    end
  )) as OwnerBU_Old, 
  upper(trim(
    case 
      when UP.Country is null then 'UNIDENTIFIED' 
      else UP.Country 
    end
  )) as OwnerCountry, 
  upper(trim(CMS.lastAuthor)) as last_update_by
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as CMS
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details as CL
  on upper(trim(CL.Document_ID)) = upper(trim(cast(CMS.Document_Id as string)))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP
  on upper(trim(UP.UserName)) = upper(trim(
    case 
      when upper(trim(CMS.created)) = 'ADMINISTRATOR' then CMS.lastAuthor 
      else CMS.created 
    end
  ))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted as S1
  on upper(trim(CMS.Document_Id)) = upper(trim(S1.document_id))
where CMS.Document_Id is not null 
  and upper(trim(cms.Document_name)) not in ('DOCUMENT NOT FOUND ON SERVER')
  -- and (
  --   (upper(trim(CMS.created)) <> 'ADMINISTRATOR' and CMS.created is not null)
  --   or (upper(trim(CMS.created)) = 'ADMINISTRATOR' and CMS.lastAuthor is not null)
  -- )

In [0]:
%sql

CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.custom_sap_bo.Controler_Finance_Audit_analysis
select distinct CT.`L1-_Name` as Manager, CL.cluster, UP.DisplayName as Report_user, UA.WEBI_Doc_ID as AccessedDoc 
from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cms_file_metadata as CMS
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details as CL
on upper(CL.Document_ID)=upper(cast(CMS.Report_ID as string))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA
on upper(CL.Document_ID) = upper(CAST(UA.WEBI_Doc_ID AS string))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP 
on upper(UP.UserName) = upper(UA.UserName)
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.control_finance_team as CT
on upper(UP.EmailAddress)=upper(CT.`L2-_email`)
where CT.`L2-_Name` is not null
union
select distinct  CT.`L1-_Name`as Manager, CL.cluster, UP.DisplayName as Report_user,  UA.WEBI_Doc_ID as AccessedDoc 
from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cms_file_metadata as CMS
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details as CL
on upper(CL.Document_ID)=upper(cast(CMS.Report_ID as string))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA
on upper(CL.Document_ID) = upper(CAST(UA.WEBI_Doc_ID AS string))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP 
on upper(UP.UserName) = upper(UA.UserName)
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.control_finance_team as CT
on upper(UP.EmailAddress)=upper(CT.`L1-_email`)
where CT.`L1-_Name` is not null
order by Manager, CL.cluster asc;


select distinct CT.`L1-_Name` as Mnger, CT.`L2-_Name` as UserName, Up.DisplayName from dataplatform01_central_dev_catalog_01.custom_sap_bo.controler_teams as CT
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP 
on UP.EmailAddress=CT.`L2-_email`
Union
select distinct CT.`L1-_Name` as UserName, CT.`L1-_Name` as UserName, Up.DisplayName from dataplatform01_central_dev_catalog_01.custom_sap_bo.controler_teams as CT
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP 
on UP.EmailAddress=CT.`L1-_email`;

-- dataplatform01_central_dev_catalog_01.custom_sap_bo.control_finance_team


select distinct  CT.`L1-_Name`,CT.`L2-_Department`, "Analysis_Result" as Analysis
 from dataplatform01_central_dev_catalog_01.custom_sap_bo.control_finance_team as CT
 left join dataplatform01_central_dev_catalog_01.custom_sap_bo.Controler_Finance_Audit_analysis as CFAA
 on CT.`L1-_Name` = CFAA.Manager
 where  CFAA.Manager is null
union
select distinct  CT.`L1-_Name`, "Profile_NOT_available" as Analysis  from 
dataplatform01_central_dev_catalog_01.custom_sap_bo.control_finance_team as CT
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP
on CT.`L2-_email`=UP.EmailAddress
where UP.EmailAddress is null
order by `L1-_Name` asc
;

select distinct UP.EmailAddress, UP.UserName
 from dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP
 where UP.BU = "GBUND-Group Buyer Underwriting"
 order by UP.BU asc;

 select distinct CT.`L1-_Name` as Manager, CL.cluster, UP.DisplayName as Report_user, UA.WEBI_Doc_ID as AccessedDoc 
from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cms_file_metadata as CMS
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details as CL
on CL.Document_ID=cast(CMS.Report_ID as string)
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA
on CL.Document_ID = CAST(UA.WEBI_Doc_ID AS string)
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP 
on UP.UserName = UA.UserName
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.control_finance_team as CT
on UP.EmailAddress=CT.`L2-_email`
where CT.`L2-_Name` is not null
and UP.BU = "GBUND-Group Buyer Underwriting";

select distinct UP.UserName, CT.`L2-_Name` from 
-- dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA
-- left join 
dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP 
-- on UP.UserName = UA.UserName
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.control_finance_team as CT
on upper(UP.EmailAddress)=upper(CT.`L2-_email`)

select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.control_finance_team as CT
where 
upper(CT.`L2-_email`)=upper("Hugh.Morgan@atradius.com")

-- CT.`L2-_Name` is not null
-- and 
-- UP.BU = "GBUND-Group Buyer Underwriting"

select Manager, count(distinct AccessedDoc) as document_cnt, count(distinct Report_user) as usre_cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.Controler_Finance_Audit_analysis as CFAA
group by Manager
order by Manager asc

select CT.`L1-_Name` as manager, count(distinct CT.`L2-_email`)+1 as users from dataplatform01_central_dev_catalog_01.custom_sap_bo.control_finance_team as CT
group by CT.`L1-_Name`
order by manager asc


In [0]:

%sql
select 
  ca1.cluster, 
  case 
    when ca1.BU is null or upper(ca1.BU) in ('LEFT ORGANIZATION', 'UNIDENTIFIED') then 'UNIDENTIFIED'
    else ca1.BU
  end as BU,
  case
    when lower(se1.Active_Schedule)='true' then True
    else False
  end as Scheduled,
  count(distinct ca1.Doc_ID) as document_cnt, 
  count(distinct ca1.UserName) as user_cnt  
from dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis ca1
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted se1
 on trim(ca1.Doc_ID) = trim(se1.document_id)
group by cluster, 
  case 
    when ca1.BU is null or upper(ca1.BU) in ('LEFT ORGANIZATION', 'UNIDENTIFIED') then 'UNIDENTIFIED'
    else ca1.BU
  end,
  se1.Active_Schedule
having count(distinct UserName) > 0
order by cluster asc, BU asc
;

select 
  ca1.cluster, 
  case 
    when ca1.OwnerBU is null or upper(ca1.OwnerBU) in ('LEFT ORGANIZATION', 'UNIDENTIFIED') then 'UNIDENTIFIED'
    else ca1.OwnerBU
  end as BU,
  case
    when lower(se1.Active_Schedule)='true' then True
    else False
  end as Scheduled,
  count(distinct ca1.Doc_ID) as document_cnt, 
  count(distinct ca1.OwnerName) as user_cnt  
from dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_owner_analysis ca1
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted se1
 on trim(ca1.Doc_ID) = trim(se1.document_id)
group by cluster, 
  case 
    when ca1.OwnerBU is null or upper(ca1.OwnerBU) in ('LEFT ORGANIZATION', 'UNIDENTIFIED') then 'UNIDENTIFIED'
    else ca1.OwnerBU
  end
  ,
  se1.Active_Schedule
having count(distinct ca1.OwnerName) > 0
order by cluster asc, BU asc
;

select distinct lastAuthor 
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
-- dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis
where not (lastAuthor rlike '^[A-Za-z]{6}[0-9]{1}$')

In [0]:
%sql 
select wcd.cluster, cms.scheduled, count(distinct uaa.WEBI_Doc_ID) as audit_cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UAA
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details as wcd
on upper(trim(uaa.webi_doc_id)) = upper(trim(wcd.Document_Id))
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms
on upper(trim(wcd.Document_Id)) = upper(trim(cms.Document_Id))
group by wcd.cluster, cms.scheduled

select cms.scheduled, count(distinct cms.Document_Id) as cnt from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms
group by cms.scheduled

select dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
select scheduled, count(distinct Report_ID) as cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cms_file_metadata
group by scheduled

select "Now" as category,  CMS_scheduled, count(distinct Report_ID) as cnt from
(
  select distinct FID.Report_ID, 
  (Case when WEBA.Object_ID is not null then "Active" end) as Audit, 
  CMS.scheduled as CMS_scheduled
  from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.sapbo_full_doc_id as FID
  left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.bo_user_access_audit as WEBA
  on upper(FID.Report_CUID) =upper(WEBA.Object_ID)
  left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as CMS
  on upper(FID.Report_ID) = upper(CMS.Document_Id)
  where upper(FID.Report_Type)="WEBI"
  order by FID.Report_ID asc
) as report
group by category, CMS_scheduled

union all 
select "Befor" as category, CMS_scheduled, count(distinct Report_ID) as cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.Webi_schedule_scan_tracker
group by category, CMS_scheduled

dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms 
left join 
select count(distinct UAA.webi_doc_id) as web_cnt, count(distinct uaa.Audit_id) as aud_cnt
from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UAA
where 

In [0]:
%sql
select cluster, count(distinct Doc_ID) as cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis
group by cluster